# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [1]:
pip install langchain-openai langchain-community tavily-python python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: c:\Users\parte\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:
import os
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.chat_models import ChatOpenAI
from langchain.tools.tavily_search import TavilySearchResults
import tiktoken
from IPython.display import Markdown, display
from  dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
from langchain.tools.tavily_search import TavilySearchResults

load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  # leave room for GPT-4 response
    return trimmed


# LangChain tool
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]
agent = initialize_agent(tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

c:\Users\parte\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'initialize_agent' from 'langchain.agents' (c:\Users\parte\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain\agents\__init__.py)

In [3]:
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
import tiktoken

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage


In [4]:
load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.2,
    top_p=0.9,
    frequency_penalty=0.1,
    presence_penalty=0.0
)

encoding = tiktoken.encoding_for_model("gpt-4o")


In [5]:
tavily_search = TavilySearchResults(max_results=5)

def safe_search(query: str) -> str:
    result = tavily_search.run(query)
    text = str(result)
    tokens = encoding.encode(text)
    return encoding.decode(tokens[:3500])


### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [18]:
prompt_1 = f"""
You are an AI assistant designed to research and summarize real-time information.

Your task is to gather the **latest advancements in generative AI applied to healthcare in 2025** using web search results.

Please output the answer in **Markdown format** with the following structure:

## 🩺 Generative AI in Healthcare — 2025 Summary

1. **Key innovations and technologies**  
   - What new tools, models, or applications have emerged?
2. **Use cases in clinical environments**  
   - How is generative AI being used by hospitals, researchers, or health systems?
3. **Impact and adoption trends**  
   - What trends or statistics show adoption, growth, or investment?
4. **Challenges and ethical considerations**  
   - What are risks, barriers, or concerns with deploying these technologies?
5. **Notable companies or initiatives**  
   - List organizations or partnerships driving change.

Be factual and concise. If specific details are uncertain or unavailable, say so clearly. Use the retrieved context to support your answer.
"""


In [19]:
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

load_dotenv()

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.2,
    top_p=0.9,
    frequency_penalty=0.1,
    presence_penalty=0.0
)

# Initialize Tavily tool
tavily_search = TavilySearchResults(
    max_results=5,
    search_depth="advanced"
)

# Bind tool to LLM (this replaces agents)
llm_with_tools = llm.bind_tools([tavily_search])


In [20]:
from langchain_core.messages import HumanMessage
from IPython.display import Markdown, display

response_1 = llm_with_tools.invoke(
    [HumanMessage(content=prompt_1)]
)

display(Markdown(response_1.content))


In [23]:
display(Markdown(response_1.content))


### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [24]:
prompt_2 = """
You are an AI assistant with access to real-time web search tools.

Task:
Identify the top emerging AI startups in 2025 and summarize their innovations.

Instructions:
- Use web search to find recent (2025) information
- For each startup, include:
  - Company name
  - Product or innovation highlight
  - Source of information
- Present the answer in clean, well-structured Markdown

Format:

## 🚀 Top Emerging AI Startups — 2025

### Startup Name
- **Innovation / Product:** …
- **Description:** …
- **Source:** …

List at least 4–6 startups.
Do not invent information. Use only retrieved data.
"""


In [27]:
from langchain_core.messages import HumanMessage, ToolMessage
from IPython.display import Markdown, display

# First model call (requests tool)
response_2 = llm_with_tools.invoke(
    [HumanMessage(content=prompt_2)]
)

messages = [HumanMessage(content=prompt_2), response_2]

# Execute Tavily tool calls
for tool_call in response_2.tool_calls:
    tool_result = tavily_search.invoke(tool_call["args"])
    messages.append(
        ToolMessage(
            content=str(tool_result),
            tool_call_id=tool_call["id"]
        )
    )

# Final model call (generates answer)
final_response_2 = llm.invoke(messages)

# Display Markdown
display(Markdown(final_response_2.content))


Hello! How can I assist you today?

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [29]:
prompt_3 = """
You are an AI assistant with access to real-time web search tools.

Task:
Compare the tech products **ChatGPT Team (by OpenAI)** and **Claude Pro (by Anthropic)**.

Instructions:
- Use web search to retrieve the most recent information
- Compare the products based on:
  - Key features
  - Pricing plans
  - User reviews or general feedback
- Do not invent details; rely only on retrieved sources
- If information is missing or unclear, state this explicitly

Present the response in **structured Markdown** using the following format:

## 🔍 Product Comparison: ChatGPT Team vs Claude Pro

### 🟦 ChatGPT Team
- **Key Features:** …
- **Pricing:** …
- **User Reviews / Feedback:** …
- **Sources:** …

### 🟨 Claude Pro
- **Key Features:** …
- **Pricing:** …
- **User Reviews / Feedback:** …
- **Sources:** …

### 📊 Summary Comparison
- Strengths of each product
- Suitable use cases
"""


In [31]:
from langchain_core.messages import HumanMessage, ToolMessage
from IPython.display import Markdown, display

# First call (tool request)
response_3 = llm_with_tools.invoke(
    [HumanMessage(content=prompt_3)]
)

messages = [HumanMessage(content=prompt_3), response_3]

# Execute Tavily tool calls
for tool_call in response_3.tool_calls:
    tool_result = tavily_search.invoke(tool_call["args"])
    messages.append(
        ToolMessage(
            content=str(tool_result),
            tool_call_id=tool_call["id"]
        )
    )

# Final response
final_response_3 = llm.invoke(messages)

display(Markdown(final_response_3.content))


## 🔍 Product Comparison: ChatGPT Team vs Claude Pro

### 🟦 ChatGPT Team
- **Key Features:**
  - Shared team workspace for collaboration.
  - Admin controls for user management.
  - Custom GPT creation and sharing within the workspace.
  - Data privacy with no training on business data by default.
  - Higher usage caps on advanced models.
- **Pricing:**
  - $30 per user per month (monthly billing).
  - $25 per user per month (annual billing).
- **User Reviews / Feedback:**
  - Generally positive feedback for collaborative features and data privacy.
  - Some concerns about potential budget issues due to user management setup allowing any team member to invite new users, which automatically adds paid seats.
- **Sources:**
  - [Eesel AI Blog](https://www.eesel.ai/blog/chatgpt-teams)
  - [WithOrb Blog](https://www.withorb.com/blog/openai-pricing)
  - [CloudEagle Blog](https://www.cloudeagle.ai/blogs/blog-chatgpt-pricing-guide)

### 🟨 Claude Pro
- **Key Features:**
  - Access to advanced Opus model for enhanced coding, reasoning, and long-context performance.
  - Priority access during peak times.
  - Integration with Google Workspace and other tools.
  - Extended context windows and support for multiple models.
- **Pricing:**
  - $20 per month (monthly billing).
  - $17 per month (annual billing, $200 upfront).
- **User Reviews / Feedback:**
  - Positive reviews for its performance in code generation and debugging.
  - Users appreciate the extended usage limits and priority access.
  - Some users find the pricing justified by the performance improvements over the free version.
- **Sources:**
  - [Intuition Labs](https://intuitionlabs.ai/articles/claude-pricing-plans-api-costs)
  - [GamsGo Blog](https://www.gamsgo.com/blog/is-claude-pro-worth-it)
  - [GetMonetizely](https://www.getmonetizely.com/articles/claude-pro-pricing-in-2025-plans-limits-and-how-it-compares)

### 📊 Summary Comparison
- **Strengths of Each Product:**
  - **ChatGPT Team:** Strong collaborative features with a focus on data privacy and admin controls, making it suitable for teams needing a shared workspace and custom GPT capabilities.
  - **Claude Pro:** Offers advanced model capabilities with priority access, making it ideal for users who require enhanced performance in tasks like coding and reasoning.

- **Suitable Use Cases:**
  - **ChatGPT Team:** Best for small to mid-sized teams that need a collaborative environment with robust admin controls and data privacy.
  - **Claude Pro:** Suitable for individual users or teams that require high-performance AI capabilities, particularly in technical fields like software development.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
